<a href="https://colab.research.google.com/github/aranavkumar/Veloura/blob/main/07_Veloura_Safety_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

# ============================================================
# PHASE 7: CONNECT GOOGLE DRIVE
# ============================================================

from google.colab import drive
import os

# Use a fresh mountpoint for Phase 7
MOUNT_DIR = "/content/drive_phase7"

os.makedirs(MOUNT_DIR, exist_ok=True)

drive.mount(MOUNT_DIR)

PROJECT_DIR = os.path.join(
    MOUNT_DIR,
    "MyDrive",
    "Veloura_Project"
)

print("=" * 55)
print("PHASE 7 DRIVE CONNECTION")
print("=" * 55)

print("Drive mounted:", os.path.exists(
    os.path.join(MOUNT_DIR, "MyDrive")
))

print("Project exists:", os.path.exists(PROJECT_DIR))

# Check Phase 6 output
INPUT_JSON = os.path.join(
    PROJECT_DIR,
    "outputs",
    "reasoning_context",
    "reasoning_context_sample_logs.json"
)

print("Phase 6 JSON exists:", os.path.exists(INPUT_JSON))

if os.path.exists(INPUT_JSON):
    print("File size:", os.path.getsize(INPUT_JSON), "bytes")
    print("\nSUCCESS: PHASE 6 OUTPUT ACCESSIBLE!")
else:
    print("\nPhase 6 output still not visible.")


Mounted at /content/drive_phase7
PHASE 7 DRIVE CONNECTION
Drive mounted: True
Project exists: True
Phase 6 JSON exists: True
File size: 25115 bytes

SUCCESS: PHASE 6 OUTPUT ACCESSIBLE!


In [ ]:

# ============================================================
# CELL 2: VELOURA SAFETY RULES ENGINE
# ============================================================

import json
import os
import pandas as pd
from datetime import datetime, timezone

# Load Phase 6 input
with open(INPUT_JSON, "r") as file:
    phase6_data = json.load(file)

# Safety rule definitions
SAFETY_RULES = {
    "MISSING_PATIENT_CONTEXT": True,
    "EXPERIMENTAL_FEATURES": True,
    "UNVALIDATED_SEGMENTATION": True,
    "MANDATORY_HUMAN_REVIEW": True,
    "AUTOMATIC_ACCEPTANCE_BLOCKED": True,
    "INDEPENDENT_DIAGNOSIS_PROHIBITED": True
}


def evaluate_safety(context):

    flags = []
    warnings = []

    patient = context["patient_context"]
    lesion = context["lesion_context"]
    constraints = context["clinical_constraints"]

    # Rule 1: Missing patient information
    missing = context.get("missing_information", [])

    if missing:
        flags.append("MISSING_PATIENT_CONTEXT")
        warnings.append(
            "Patient context is incomplete: "
            + ", ".join(missing)
        )

    # Rule 2: Experimental feature evidence
    feature_status = lesion.get("feature_status")

    if feature_status == "EXPERIMENTAL_UNVALIDATED":
        flags.append("EXPERIMENTAL_FEATURES")
        warnings.append(
            "Lesion features are experimental and unvalidated."
        )

    # Rule 3: Unvalidated segmentation
    source_warnings = context.get(
        "source_evidence", {}
    ).get("warnings", [])

    if "SEGMENTATION_NOT_VALIDATED_AGAINST_EXPERT_MASKS" in source_warnings:
        flags.append("UNVALIDATED_SEGMENTATION")
        warnings.append(
            "Segmentation has not been validated against expert masks."
        )

    # Rule 4: Mandatory human review
    if (
        lesion.get("analysis_status") == "REVIEW_REQUIRED"
        or constraints.get("requires_review") is True
    ):
        flags.append("MANDATORY_HUMAN_REVIEW")
        warnings.append(
            "Human review is required before further clinical action."
        )

    # Rule 5: Automatic acceptance restriction
    if constraints.get("automatic_acceptance") is not False:
        flags.append("AUTOMATIC_ACCEPTANCE_RESTRICTION")
        warnings.append(
            "Automatic acceptance is not permitted."
        )

    # Rule 6: Independent diagnosis prohibited
    if constraints.get("independent_diagnosis") is not False:
        flags.append("DIAGNOSTIC_AUTHORITY_RESTRICTION")
        warnings.append(
            "The agent cannot independently diagnose the lesion."
        )

    # Conservative workflow decision
    safety_decision = (
        "HUMAN_REVIEW_REQUIRED"
        if flags
        else "NO_AUTOMATED_CLINICAL_CLEARANCE"
    )

    return {
        "image_id": context["image_id"],
        "agent": "Veloura Safety Agent",
        "safety_decision": safety_decision,
        "safety_flags": flags,
        "warnings": warnings,

        "clinical_safeguards": {
            "independent_diagnosis": False,
            "automatic_acceptance": False,
            "human_review_required": True,
            "experimental_evidence_validated": False
        },

        "source_analysis_status": lesion.get(
            "analysis_status"
        ),

        "source_feature_status": feature_status,

        "evaluated_at": datetime.now(
            timezone.utc
        ).isoformat()
    }


# Evaluate every Phase 6 record
safety_results = [
    evaluate_safety(record)
    for record in phase6_data
]

# Display results
print("=" * 65)
print("VELOURA SAFETY RULES ENGINE")
print("=" * 65)

print("Records evaluated:", len(safety_results))

for result in safety_results:

    print("\nIMAGE:", result["image_id"])
    print("Decision:", result["safety_decision"])
    print("Safety flags:", result["safety_flags"])
    print("Warnings:", len(result["warnings"]))

# Safety summary
flag_summary = pd.DataFrame([
    {
        "image_id": result["image_id"],
        "safety_decision": result["safety_decision"],
        "safety_flag_count": len(result["safety_flags"]),
        "human_review_required": result[
            "clinical_safeguards"
        ]["human_review_required"],
        "automatic_acceptance": result[
            "clinical_safeguards"
        ]["automatic_acceptance"]
    }
    for result in safety_results
])

print("\n" + "=" * 65)
print("SAFETY SUMMARY")
print("=" * 65)

print(flag_summary.to_string(index=False))

print("\nSUCCESS: SAFETY RULES ENGINE EXECUTED!")


VELOURA SAFETY RULES ENGINE
Records evaluated: 6

IMAGE: ISIC_0033038
Decision: HUMAN_REVIEW_REQUIRED
Safety flags: ['MISSING_PATIENT_CONTEXT', 'EXPERIMENTAL_FEATURES', 'UNVALIDATED_SEGMENTATION', 'MANDATORY_HUMAN_REVIEW']
Warnings: 4

IMAGE: ISIC_0025481
Decision: HUMAN_REVIEW_REQUIRED
Safety flags: ['MISSING_PATIENT_CONTEXT', 'EXPERIMENTAL_FEATURES', 'UNVALIDATED_SEGMENTATION', 'MANDATORY_HUMAN_REVIEW']
Warnings: 4

IMAGE: ISIC_0034119
Decision: HUMAN_REVIEW_REQUIRED
Safety flags: ['MISSING_PATIENT_CONTEXT', 'EXPERIMENTAL_FEATURES', 'UNVALIDATED_SEGMENTATION', 'MANDATORY_HUMAN_REVIEW']
Warnings: 4

IMAGE: ISIC_0031437
Decision: HUMAN_REVIEW_REQUIRED
Safety flags: ['MISSING_PATIENT_CONTEXT', 'EXPERIMENTAL_FEATURES', 'UNVALIDATED_SEGMENTATION', 'MANDATORY_HUMAN_REVIEW']
Warnings: 4

IMAGE: ISIC_0031157
Decision: HUMAN_REVIEW_REQUIRED
Safety flags: ['MISSING_PATIENT_CONTEXT', 'EXPERIMENTAL_FEATURES', 'UNVALIDATED_SEGMENTATION', 'MANDATORY_HUMAN_REVIEW']
Warnings: 4

IMAGE: ISIC_0028145


In [ ]:
# ============================================================
# CELL 3: SAFETY DECISION & COORDINATOR INTEGRATION
# ============================================================

import json
from datetime import datetime, timezone

# Index safety results by image ID
safety_lookup = {
    item["image_id"]: item
    for item in safety_results
}

coordinator_results = []

for context in phase6_data:

    image_id = context["image_id"]
    safety = safety_lookup[image_id]

    # Conservative workflow gate
    workflow_allowed = False

    coordinator_record = {
        "image_id": image_id,
        "agent": "Veloura Coordinator",
        "workflow_status": "BLOCKED_PENDING_HUMAN_REVIEW",
        "safety_decision": safety["safety_decision"],

        "safety_gate": {
            "passed": False,
            "decision": "HUMAN_REVIEW_REQUIRED",
            "flags": safety["safety_flags"],
            "warnings": safety["warnings"]
        },

        "workflow_permissions": {
            "continue_to_human_review": True,
            "allow_automated_clinical_clearance": False,
            "allow_independent_diagnosis": False,
            "allow_automatic_acceptance": False
        },

        "source_evidence": {
            "analysis_status": context[
                "lesion_context"
            ]["analysis_status"],
            "feature_status": context[
                "lesion_context"
            ]["feature_status"],
            "missing_information": context[
                "missing_information"
            ]
        },

        "coordinator_action": (
            "ESCALATE_TO_CLINICIAN"
        ),

        "workflow_note": (
            "Automated workflow is restricted. "
            "A qualified clinician must review the case. "
            "This agent does not provide a diagnosis."
        ),

        "evaluated_at": datetime.now(
            timezone.utc
        ).isoformat()
    }

    coordinator_results.append(coordinator_record)


# Display coordinator decisions
print("=" * 70)
print("VELOURA COORDINATOR — SAFETY GATE INTEGRATION")
print("=" * 70)

for record in coordinator_results:

    print("\nIMAGE:", record["image_id"])
    print("Workflow:", record["workflow_status"])
    print("Safety gate passed:", record["safety_gate"]["passed"])
    print("Coordinator action:", record["coordinator_action"])
    print(
        "Automatic acceptance:",
        record["workflow_permissions"][
            "allow_automatic_acceptance"
        ]
    )

print("\nRecords integrated:", len(coordinator_results))
print("SUCCESS: COORDINATOR SAFETY GATE CREATED!")

VELOURA COORDINATOR — SAFETY GATE INTEGRATION

IMAGE: ISIC_0033038
Workflow: BLOCKED_PENDING_HUMAN_REVIEW
Safety gate passed: False
Coordinator action: ESCALATE_TO_CLINICIAN
Automatic acceptance: False

IMAGE: ISIC_0025481
Workflow: BLOCKED_PENDING_HUMAN_REVIEW
Safety gate passed: False
Coordinator action: ESCALATE_TO_CLINICIAN
Automatic acceptance: False

IMAGE: ISIC_0034119
Workflow: BLOCKED_PENDING_HUMAN_REVIEW
Safety gate passed: False
Coordinator action: ESCALATE_TO_CLINICIAN
Automatic acceptance: False

IMAGE: ISIC_0031437
Workflow: BLOCKED_PENDING_HUMAN_REVIEW
Safety gate passed: False
Coordinator action: ESCALATE_TO_CLINICIAN
Automatic acceptance: False

IMAGE: ISIC_0031157
Workflow: BLOCKED_PENDING_HUMAN_REVIEW
Safety gate passed: False
Coordinator action: ESCALATE_TO_CLINICIAN
Automatic acceptance: False

IMAGE: ISIC_0028145
Workflow: BLOCKED_PENDING_HUMAN_REVIEW
Safety gate passed: False
Coordinator action: ESCALATE_TO_CLINICIAN
Automatic acceptance: False

Records integrate

In [ ]:
# ============================================================
# CELL 4: PHASE 7 EXPORT & FINAL VERIFICATION
# ============================================================

import os
import json
import pandas as pd

# Use the confirmed Phase 7 Google Drive location
PROJECT_DIR = "/content/drive_phase7/MyDrive/Veloura_Project"

OUTPUT_DIR = os.path.join(
    PROJECT_DIR, "outputs", "safety_agent"
)

REPORT_DIR = os.path.join(
    PROJECT_DIR, "reports", "safety_agent"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(REPORT_DIR, exist_ok=True)

# ------------------------------------------------------------
# 1. EXPORT SAFETY AGENT RESULTS
# ------------------------------------------------------------

SAFETY_JSON = os.path.join(
    OUTPUT_DIR, "safety_agent_sample_logs.json"
)

with open(SAFETY_JSON, "w") as file:
    json.dump(safety_results, file, indent=4)


# ------------------------------------------------------------
# 2. EXPORT COORDINATOR INTEGRATION RESULTS
# ------------------------------------------------------------

COORDINATOR_JSON = os.path.join(
    OUTPUT_DIR, "coordinator_integration_sample_logs.json"
)

with open(COORDINATOR_JSON, "w") as file:
    json.dump(coordinator_results, file, indent=4)


# ------------------------------------------------------------
# 3. CREATE COMBINED SUMMARY REPORT
# ------------------------------------------------------------

summary_rows = []

for safety, coordinator in zip(
    safety_results, coordinator_results
):

    summary_rows.append({
        "image_id": safety["image_id"],
        "safety_decision": safety["safety_decision"],
        "safety_flag_count": len(safety["safety_flags"]),
        "safety_flags": " | ".join(safety["safety_flags"]),
        "workflow_status": coordinator["workflow_status"],
        "coordinator_action": coordinator["coordinator_action"],
        "safety_gate_passed": coordinator["safety_gate"]["passed"],
        "human_review_required": coordinator[
            "workflow_permissions"
        ]["continue_to_human_review"],
        "automatic_acceptance": coordinator[
            "workflow_permissions"
        ]["allow_automatic_acceptance"],
        "independent_diagnosis": coordinator[
            "workflow_permissions"
        ]["allow_independent_diagnosis"]
    })

summary_df = pd.DataFrame(summary_rows)

CSV_REPORT = os.path.join(
    REPORT_DIR, "safety_agent_sample_summary.csv"
)

summary_df.to_csv(CSV_REPORT, index=False)


# ------------------------------------------------------------
# 4. RELOAD EXPORTED FILES
# ------------------------------------------------------------

with open(SAFETY_JSON, "r") as file:
    exported_safety = json.load(file)

with open(COORDINATOR_JSON, "r") as file:
    exported_coordinator = json.load(file)

exported_summary = pd.read_csv(CSV_REPORT)


# ------------------------------------------------------------
# 5. FINAL VERIFICATION TESTS
# ------------------------------------------------------------

expected_ids = {
    record["image_id"] for record in phase6_data
}

expected_flags = {
    "MISSING_PATIENT_CONTEXT",
    "EXPERIMENTAL_FEATURES",
    "UNVALIDATED_SEGMENTATION",
    "MANDATORY_HUMAN_REVIEW"
}

tests = {
    "Six safety records exported":
        len(exported_safety) == 6,

    "Six coordinator records exported":
        len(exported_coordinator) == 6,

    "Six CSV summary records exported":
        len(exported_summary) == 6,

    "Safety image IDs match Phase 6":
        {r["image_id"] for r in exported_safety} == expected_ids,

    "Coordinator image IDs match Phase 6":
        {r["image_id"] for r in exported_coordinator} == expected_ids,

    "All four expected safety flags preserved":
        all(
            expected_flags.issubset(set(r["safety_flags"]))
            for r in exported_safety
        ),

    "All decisions require human review":
        all(
            r["safety_decision"] == "HUMAN_REVIEW_REQUIRED"
            for r in exported_safety
        ),

    "All coordinator workflows remain blocked":
        all(
            r["workflow_status"] == "BLOCKED_PENDING_HUMAN_REVIEW"
            for r in exported_coordinator
        ),

    "All cases routed to clinician":
        all(
            r["coordinator_action"] == "ESCALATE_TO_CLINICIAN"
            for r in exported_coordinator
        ),

    "Safety gates remain closed":
        all(
            r["safety_gate"]["passed"] is False
            for r in exported_coordinator
        ),

    "Automatic acceptance remains disabled":
        all(
            r["workflow_permissions"]["allow_automatic_acceptance"] is False
            for r in exported_coordinator
        ),

    "Independent diagnosis remains prohibited":
        all(
            r["workflow_permissions"]["allow_independent_diagnosis"] is False
            for r in exported_coordinator
        )
}

print("=" * 70)
print("VELOURA PHASE 7 — FINAL VERIFICATION")
print("=" * 70)

for test_name, passed in tests.items():
    print(
        f"[{'PASS' if passed else 'FAIL'}] {test_name}"
    )

print("\nVerification:", sum(tests.values()), "/", len(tests))

print("\nEXPORTED FILES")
print("Safety JSON:", SAFETY_JSON)
print("Coordinator JSON:", COORDINATOR_JSON)
print("Summary CSV:", CSV_REPORT)

print("\nSUMMARY PREVIEW")
print(summary_df.to_string(index=False))

if all(tests.values()):
    print("\nSUCCESS: PHASE 7 COMPLETED AND VERIFIED!")
else:
    print("\nWARNING: SOME VERIFICATION TESTS FAILED.")

VELOURA PHASE 7 — FINAL VERIFICATION
[PASS] Six safety records exported
[PASS] Six coordinator records exported
[PASS] Six CSV summary records exported
[PASS] Safety image IDs match Phase 6
[PASS] Coordinator image IDs match Phase 6
[PASS] All four expected safety flags preserved
[PASS] All decisions require human review
[PASS] All coordinator workflows remain blocked
[PASS] All cases routed to clinician
[PASS] Safety gates remain closed
[PASS] Automatic acceptance remains disabled
[PASS] Independent diagnosis remains prohibited

Verification: 12 / 12

EXPORTED FILES
Safety JSON: /content/drive_phase7/MyDrive/Veloura_Project/outputs/safety_agent/safety_agent_sample_logs.json
Coordinator JSON: /content/drive_phase7/MyDrive/Veloura_Project/outputs/safety_agent/coordinator_integration_sample_logs.json
Summary CSV: /content/drive_phase7/MyDrive/Veloura_Project/reports/safety_agent/safety_agent_sample_summary.csv

SUMMARY PREVIEW
    image_id       safety_decision  safety_flag_count        